# Day 2 homework — chunking and intelligent processing

Same topics as **`course/Day_02_Chunking_and_Intelligent_Processing.ipynb`**, but **`project/`** uses **different repos** than the course (Evidently): here we load **`pydantic/pydantic`** docs on `main`.

```bash
uv add requests python-frontmatter openai tqdm
uv add --dev jupyter
uv run jupyter notebook
```

## Small vs large documents

Small pages can be indexed whole. Large docs should be **chunked** so you stay within **token** / **cost** limits, keep **relevance**, and avoid dumping irrelevant text into the model.

## Today’s tasks

1. Sliding-window (character) chunking with overlap  
2. Paragraph + **Markdown section** splitting  
3. Optional **LLM** chunking (needs `OPENAI_API_KEY` or another provider)  

Then apply what works to **your** chosen repo from Day 1.

## Load data (homework repo — not Evidently)

`pydantic/pydantic` default branch is **`main`** (same zip URL pattern as Day 1 homework).

In [1]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"Error processing {fn}: {e}")
    zf.close()
    return out


docs = read_repo_data("pydantic", "pydantic")
len(docs)

98

## Pick a long document for demos

Course Day 2 hard-codes Evidently **index 45**. Here we pick the **longest** `content` so the notebook stays stable if the list order changes.

In [2]:
def pick_long_doc_index(documents):
    best_i, best_n = 0, -1
    for i, d in enumerate(documents):
        n = len(d.get("content") or "")
        if n > best_n:
            best_i, best_n = i, n
    return best_i, best_n


demo_idx, n_chars = pick_long_doc_index(docs)
demo_idx, n_chars, docs[demo_idx].get("title") or docs[demo_idx].get("filename")

(2, 319104, 'pydantic-main/HISTORY.md')

## 1. Simple sliding-window chunking

In [3]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= n:
            break
    return result


text_demo = docs[demo_idx]["content"]
chunks_demo = sliding_window(text_demo, 2000, 1000)
len(chunks_demo)

319

In [4]:
chunks_sliding = []

for doc in docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    chunks_sliding.extend(chunks)

len(chunks_sliding)

1035

## 2. Paragraphs

In [5]:
import re

text = docs[demo_idx]["content"]
paragraphs = re.split(r"\n\s*\n", text.strip())
len(paragraphs)

827

## 3. Sections (`##` level 2)

In [6]:
import re


def split_markdown_by_level(text, level=2):
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)
    sections = []
    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i + 1]
        header = header.strip()
        content = ""
        if i + 2 < len(parts):
            content = parts[i + 2].strip()
        if content:
            section = f"{header}\n\n{content}"
        else:
            section = header
        sections.append(section)
    return sections


sections = split_markdown_by_level(text, level=2)
len(sections)

186

In [7]:
chunks_sections = []

for doc in docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc["section"] = section
        chunks_sections.append(section_doc)

len(chunks_sections)

565

## 4. Intelligent chunking (LLM)

Put `OPENAI_API_KEY=...` in **`ai_hero/.env`** (loaded automatically) or set the env var before starting Jupyter. **Start with `MAX_DOCS_FOR_LLM = 1`** — full-repo runs are expensive.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "Set OPENAI_API_KEY: create ai_hero/.env with OPENAI_API_KEY=sk-... or export the variable before Jupyter."
    )

openai_client = OpenAI()


def llm(prompt, model="gpt-4o-mini"):
    response = openai_client.responses.create(model=model, input=prompt)
    return response.output_text


prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    parts = response.split("---")
    return [s.strip() for s in parts if s.strip()]

ModuleNotFoundError: No module named 'openai'

In [ ]:
from tqdm.auto import tqdm

MAX_DOCS_FOR_LLM = 1

chunks_llm = []
for doc in tqdm(docs[:MAX_DOCS_FOR_LLM]):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    for section in intelligent_chunking(doc_content):
        row = doc_copy.copy()
        row["section"] = section
        chunks_llm.append(row)

len(chunks_llm)

## Choose a strategy

Start with **sliding window + overlap**, then **sections** if headings are reliable, then **LLM** only if simpler methods fail evaluations.

## Homework

- For **your** Day 1 repo: run these chunkers, inspect outputs, pick what fits.  
- Post progress (LinkedIn / X).  

[Course signup](https://alexeygrigorev.com/aihero/) · [DataTalks.Club Slack — `#course-ai-hero`](https://datatalks.club/)